# Modulo 3 · Lezione 3 — Ricerca ed elaborazione del testo

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | `grep` | Cercare testo dentro i file |
| 2 | `find` | Cercare file nel sistema |
| 3 | Pipe `\|` | Concatenare comandi |
| 4 | Redirezioni `>`, `>>`, `<`, `2>` | Catturare e reindirizzare output |
| 5 | `wc`, `sort`, `uniq`, `cut` | Strumenti di elaborazione del testo |
| 6 | Pratica | Analizzare log e dati dal terminale |

> 💡 **Idea centrale:** Il vero potere del terminale non sta nei singoli comandi — sta nel **combinarli**. Con pipe e redirezioni, costruisci pipeline di elaborazione che fanno in una riga ciò che uno script lungo farebbe in venti.

---
## Preparare l'ambiente di demo

In [ ]:
import os, shutil
from pathlib import Path

DEMO = Path.home() / "busqueda_demo"
if DEMO.exists(): shutil.rmtree(DEMO)
(DEMO / "logs").mkdir(parents=True)
(DEMO / "src" / "modelos").mkdir(parents=True)
(DEMO / "datos").mkdir()
os.chdir(DEMO)

# Log di addestramento
import math, random
random.seed(42)
lineas_log = []
for i in range(1, 31):
    loss = 2.0 * math.exp(-i * 0.08) + random.uniform(-0.02, 0.02)
    acc  = 1 - loss / 2.2
    lineas_log.append(f"[INFO] Epoch {i:02d}/30 loss={loss:.4f} acc={acc:.4f}")
    if i % 5 == 0:
        lineas_log.append(f"[WARNING] val_loss maggiore di train_loss all'epoch {i}")
    if i == 15:
        lineas_log.append("[ERROR] NaN rilevato nel gradiente — epoch 15")
Path("logs/entrenamiento.log").write_text("\n".join(lineas_log) + "\n", encoding="utf-8")

# File Python
Path("src/main.py").write_text("""\
import modelo
import preprocesado

def main():
    datos = preprocesado.cargar("datos/train.csv")
    m = modelo.Modelo(capas=[64, 32])
    m.entrenar(datos)
    print("Addestramento completato")

if __name__ == "__main__":
    main()
""", encoding="utf-8")

Path("src/modelos/red_neuronal.py").write_text("""\
class RedNeuronal:
    def __init__(self, capas):
        self.capas = capas
        self.pesos = {}

    def entrenar(self, datos, epochs=10, lr=0.001):
        for epoch in range(epochs):
            loss = self._paso_entrenamiento(datos)
            print(f"Epoch {epoch}: loss={loss:.4f}")

    def predecir(self, x):
        return x  # placeholder

    def _paso_entrenamiento(self, datos):
        return 0.5  # placeholder
""", encoding="utf-8")

# CSV con dati
Path("datos/ventas.csv").write_text(
    "mes,producto,ventas,region\n" +
    "\n".join([
        f"{mes},{prod},{v},{'Norte' if i%2==0 else 'Sur'}"
        for i, (mes, prod, v) in enumerate([
            ("Ene","A",100),("Ene","B",200),("Feb","A",150),
            ("Feb","B",250),("Mar","A",175),("Mar","B",300),
            ("Abr","A",200),("Abr","B",350),("May","A",225),
            ("May","B",400)
        ])
    ]) + "\n",
    encoding="utf-8"
)

print(f"Demo in: {DEMO}")
!ls -R

---
## 1. `grep` — Cercare testo nei file

`grep` (*global regular expression print*) cerca righe che corrispondono a un pattern.

```bash
grep "pattern" file          # cercare in un file
grep -i "pattern" file       # ignorare maiuscole/minuscole
grep -n "pattern" file       # mostrare il numero di riga
grep -r "pattern" directory/   # cercare ricorsivamente
grep -l "pattern" directory/   # solo i nomi dei file con corrispondenze
grep -v "pattern" file       # righe che NON corrispondono (invertire)
grep -c "pattern" file       # contare le corrispondenze
grep -E "pattern" file       # espressioni regolari estese
grep -A 2 "pattern" file     # 2 righe dopo ogni corrispondenza
grep -B 2 "pattern" file     # 2 righe prima
grep -C 2 "pattern" file     # 2 righe di contesto (prima e dopo)
```

In [ ]:
# Cercare errori nel log
print("=== Righe con ERROR ===")
!grep "ERROR" logs/entrenamiento.log

print("\n=== Righe con WARNING (con numero di riga) ===")
!grep -n "WARNING" logs/entrenamiento.log

In [ ]:
# Contare le corrispondenze
print("=== Quante righe INFO ci sono ===")
!grep -c "INFO" logs/entrenamiento.log

# Invertire: righe che NON sono INFO
print("\n=== Righe che NON sono INFO ===")
!grep -v "\[INFO\]" logs/entrenamiento.log

# Cercare con contesto
print("\n=== ERROR con 2 righe di contesto ===")
!grep -C 2 "ERROR" logs/entrenamiento.log

In [ ]:
# grep -E: espressioni regolari estese
print("=== Epoch dove acc > 0.8 (regex) ===")
!grep -E 'acc=0\.[89][0-9]+' logs/entrenamiento.log

# Cercare nel codice sorgente
print("\n=== Tutte le funzioni def nel codice ===")
!grep -rn "def " src/

print("\n=== File che importano qualcosa ===")
!grep -rl "^import " src/

---
## 2. `find` — Cercare file nel sistema

Mentre `grep` cerca **dentro** i file, `find` cerca **file** in base alle loro proprietà.

```bash
find . -name "*.py"              # file .py nella directory attuale
find . -name "*.py" -type f      # assicurarsi che siano file (non dir)
find . -type d                   # solo directory
find . -mtime -7                 # modificati negli ultimi 7 giorni
find . -mtime +30                # modificati più di 30 giorni fa
find . -size +1M                 # file maggiori di 1 MB
find . -size -100k               # file minori di 100 KB
find . -name "*.tmp" -delete     # cercare ed eliminare in un passaggio
find . -name "*.py" -exec wc -l {} \;  # eseguire un comando su ogni risultato
find . -empty                    # file o directory vuoti
find . -perm 644                 # file con permessi esatti 644
```

In [ ]:
# Cercare tutti i .py
print("=== File .py ===")
!find . -name "*.py" -type f

print("\n=== Directory ===")
!find . -type d

print("\n=== File maggiori di 500 byte ===")
!find . -type f -size +500c   # c = byte

In [ ]:
# -exec: eseguire un comando su ogni risultato
print("=== Righe in ogni .py ===")
!find . -name "*.py" -exec wc -l {} \;

# Creare alcuni file temporanei per esercitarsi con -delete
!touch temp1.tmp temp2.tmp logs/debug.tmp
print("\n=== .tmp prima di pulire ===")
!find . -name "*.tmp"

!find . -name "*.tmp" -delete
print("\n=== .tmp dopo aver pulito ===")
!find . -name "*.tmp" || echo "(nessuno)"

---
## 3. Pipe `|` — Concatenare comandi

Il **pipe** (`|`) collega l'output di un comando all'input del successivo. È il meccanismo più potente del terminale.

```
comando1 | comando2 | comando3

L'output di comando1 alimenta comando2,
il cui output alimenta comando3.
```

```bash
# Esempi progressivi:
cat log.txt | grep ERROR               # filtrare errori
cat log.txt | grep ERROR | wc -l       # contare errori
ls -la | grep ".py"                    # elencare solo .py
cat datos.csv | cut -d',' -f3 | sort   # estrarre colonna e ordinare
ps aux | grep python                   # processi Python attivi
find . -name "*.py" | xargs wc -l      # righe di tutti i .py
```

In [ ]:
# Pipeline semplice: filtrare e contare
print("=== Quanti errori/warning ci sono? ===")
!cat logs/entrenamiento.log | grep -E "ERROR|WARNING" | wc -l

# Pipeline intermedia: top 5 di qualcosa
print("\n=== Le 5 funzioni più lunghe (per righe) ===")
!find . -name "*.py" | xargs grep -c "def " 2>/dev/null | sort -t: -k2 -rn | head -5

# Contare parole uniche nei log
print("\n=== Livelli di log unici ===")
!cat logs/entrenamiento.log | grep -oE '\[\w+\]' | sort | uniq -c | sort -rn

In [ ]:
# Pipeline con cut: estrarre colonne dal CSV
print("=== Solo la colonna 'ventas' del CSV ===")
!cat datos/ventas.csv | cut -d',' -f3

print("\n=== Vendite ordinate dalla più alta alla più bassa ===")
!cat datos/ventas.csv | tail -n +2 | cut -d',' -f3 | sort -rn | head -5

print("\n=== Regioni uniche ===")
!cat datos/ventas.csv | cut -d',' -f4 | sort | uniq

---
## 4. Redirezioni — Catturare e reindirizzare

```bash
comando > file.txt       # reindirizzare stdout a file (sovrascrive)
comando >> file.txt      # aggiungere alla fine del file (append)
comando < file.txt       # leggere stdin da file
comando 2> errori.txt      # reindirizzare stderr a file
comando 2>&1                # reindirizzare stderr nello stesso posto di stdout
comando > tutto.txt 2>&1     # catturare sia stdout che stderr
comando > /dev/null         # scartare l'output
comando 2> /dev/null        # sopprimere gli errori
```

```
CANALI STANDARD:
  0 → stdin  (ingresso standard)
  1 → stdout (uscita standard)  ← > e >>
  2 → stderr (errori)           ← 2>
```

In [ ]:
# > sovrascrive, >> aggiunge
!echo "Prima riga" > resultado.txt
!echo "Seconda riga" >> resultado.txt
!echo "Terza riga" >> resultado.txt
print("Contenuto di resultado.txt:")
!cat resultado.txt

# Catturare l'output di una pipeline
!grep "ERROR" logs/entrenamiento.log > errores_filtrados.txt
print("\nErrori catturati:")
!cat errores_filtrados.txt

In [ ]:
# 2> per catturare errori (stderr)
!ls archivo_inexistente.txt 2> error_ls.txt
print("Errore catturato in error_ls.txt:")
!cat error_ls.txt

# Sopprimere errori con /dev/null
print("\nCercando *.xyz (errore silenziato):")
!find . -name "*.xyz" 2>/dev/null || echo "(nessun risultato)"

# Catturare TUTTO (stdout + stderr) nello stesso file
!find . -type f > todo_busqueda.txt 2>&1
print("\nTutto catturato:")
!cat todo_busqueda.txt

---
## 5. Strumenti di elaborazione del testo

```bash
wc -l file          # contare righe
wc -w file          # contare parole
wc -c file          # contare byte

sort file           # ordinare righe alfabeticamente
sort -n file        # ordinare numericamente
sort -r file        # ordine inverso
sort -k2 file       # ordinare per campo 2
sort -t',' -k3 -n      # CSV: ordinare per colonna 3 numericamente

uniq file           # eliminare righe duplicate consecutive*
uniq -c file        # contare le ripetizioni
uniq -d file        # solo le duplicate

cut -d',' -f1,3        # estrarre colonne 1 e 3 (delimitatore ,)
cut -c1-10             # estrarre caratteri da 1 a 10

# * uniq elimina solo i duplicati CONSECUTIVI → serve sempre sort prima
sort file | uniq    # pattern corretto per eliminare i duplicati
```

In [ ]:
# sort con CSV
print("=== ventas.csv ordinato per vendite (colonna 3, numerico) ===")
!head -n 1 datos/ventas.csv    # intestazione
!tail -n +2 datos/ventas.csv | sort -t',' -k3 -rn

print("\n=== Prodotti unici ===")
!cut -d',' -f2 datos/ventas.csv | sort | uniq

print("\n=== Conteggio occorrenze per mese ===")
!cut -d',' -f1 datos/ventas.csv | tail -n +2 | sort | uniq -c

In [ ]:
# Pipeline complessa: analisi del log in una riga
print("=== Top 3 epoch con acc maggiore (dal log) ===")
!grep '\[INFO\]' logs/entrenamiento.log \
    | grep -oE 'acc=[0-9.]+' \
    | cut -d= -f2 \
    | sort -rn \
    | head -3

print("\n=== Riepilogo: quante righe per livello ===")
!cat logs/entrenamiento.log \
    | grep -oE '\[(INFO|WARNING|ERROR)\]' \
    | sort | uniq -c | sort -rn

---
## 6. Pratica — Analizzare log e dati

---

### Esercizio 1 — Cercare nel codice sorgente

**Consegna:** Usando `grep` e `find` (senza Python), rispondi a queste domande sulla directory `~/busqueda_demo/src/`:

1. Quante funzioni (`def`) ci sono in totale?
2. Quali file importano qualche modulo?
3. In quale riga appare la parola `epochs`?
4. Quanti file `.py` ci sono in totale?
5. Mostra tutte le classi definite (`class`).

In [ ]:
# ✏️ IL TUO CODICE QUI
import os
from pathlib import Path
os.chdir(Path.home() / "busqueda_demo")


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```bash
# 1. Totale funzioni def
grep -r "def " src/ | wc -l

# 2. File che importano moduli
grep -rl "^import \|^from " src/

# 3. Riga dove appare 'epochs'
grep -rn "epochs" src/

# 4. Numero di .py
find src/ -name "*.py" -type f | wc -l

# 5. Classi definite
grep -rn "^class " src/
```
</details>

---
### Esercizio 2 — Pipeline di analisi

**Consegna:** Costruisci ogni pipeline in una sola riga di terminale:

1. Mostra le 5 righe del log con il valore di `loss` più alto.
2. Conta quante epoch hanno avuto accuracy superiore a 0.75.
3. Estrai solo i valori di `acc` dal log, convertili in lista ordinata dal più alto al più basso.
4. Dal CSV delle vendite, mostra il mese con il totale vendite più alto (somma per mese).
5. Mostra i file modificati nelle ultime 24 ore (usa `find -mtime`).

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```bash
# 1. Top 5 righe con loss maggiore
grep -oE 'loss=[0-9.]+' logs/entrenamiento.log | \
  cut -d= -f2 | sort -rn | head -5

# 2. Epoch con acc > 0.75
grep -E 'acc=0\.[789][0-9]*' logs/entrenamiento.log | wc -l

# 3. Valori di acc ordinati
grep -oE 'acc=[0-9.]+' logs/entrenamiento.log | \
  cut -d= -f2 | sort -rn

# 4. Mese con il totale vendite più alto
# (richiede awk, che vediamo brevemente qui)
tail -n +2 datos/ventas.csv | \
  awk -F',' '{sum[$1]+=$3} END {for(m in sum) print sum[m], m}' | \
  sort -rn | head -1

# 5. File modificati nelle ultime 24 ore
find . -type f -mtime -1
```

**Nota su `awk`:** È uno strumento di elaborazione testo molto potente che meriterebbe un modulo a parte. In questo corso lo usiamo puntualmente per operazioni aritmetiche sulle colonne che `cut` non può fare.
</details>

---
### Esercizio 3 — Redirezioni

**Consegna:** Esercitati con le redirezioni:

1. Salva in `informe_errores.txt` tutte le righe di errore e warning del log.
2. Esegui `ls directory_inesistente` e cattura l'errore in `error.txt`.
3. Esegui un `find` che mescoli risultati normali ed errori; salva TUTTO in `busqueda_completa.txt`.
4. Aggiungi una riga di riepilogo alla fine di `informe_errores.txt` con la data attuale: `Generato: YYYY-MM-DD`.
5. Verifica il contenuto finale di `informe_errores.txt`.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```bash
# 1. Salvare errori e warning
grep -E 'ERROR|WARNING' logs/entrenamiento.log > informe_errores.txt

# 2. Catturare errore di ls
ls directorio_que_no_existe 2> error.txt
cat error.txt

# 3. find con stdout + stderr
find / -name "*.py" -maxdepth 5 > busqueda_completa.txt 2>&1
wc -l busqueda_completa.txt

# 4. Aggiungere riga di data
echo "Generato: $(date +%Y-%m-%d)" >> informe_errores.txt

# 5. Verificare
cat informe_errores.txt
```

**`$(...)` è la sostituzione di comando:** il risultato del comando interno viene inserito come testo. `$(date +%Y-%m-%d)` produce la data attuale formattata. Lo vediamo in dettaglio nella Lezione 4 (Bash scripting).
</details>

---
### Esercizio 4 — Ricerca avanzata con find

**Consegna:** Usando solo `find` (con `-exec` o `-delete` dove serve):

1. Trova tutti i file vuoti in `~/busqueda_demo/`.
2. Trova tutte le directory vuote.
3. Trova file con permessi di scrittura per «altri» (pericolosi) con `-perm /o+w`.
4. Cerca file `.log` modificati negli ultimi due giorni.
5. Elenca tutti i file con la loro dimensione usando `-exec ls -lh {} \;`.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```bash
# 1. File vuoti
find ~/busqueda_demo -type f -empty

# 2. Directory vuote
find ~/busqueda_demo -type d -empty

# 3. File con o+w (scrittura per altri)
find ~/busqueda_demo -type f -perm /o+w

# 4. Log modificati negli ultimi 2 giorni
find ~/busqueda_demo -name "*.log" -mtime -2

# 5. Tutti i file con dimensione
find ~/busqueda_demo -type f -exec ls -lh {} \;
```
</details>

---
### Esercizio 5 (sfida) — Analizzatore di log con Python + subprocess

**Consegna:** Scrivi la funzione `analizar_logs(directorio)` che usi `subprocess` per eseguire pipeline di shell e generi un report completo della directory dei log:

- Totale righe in tutti i `.log`.
- Ripartizione per livello (INFO / WARNING / ERROR) con percentuali.
- I tre valori di `loss` più alti trovati in tutti i log.
- File di log con errori (nome + numero di errori).
- Salvare il report in `informe_logs.txt`.

In [ ]:
# ✏️ IL TUO CODICE QUI
import subprocess
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import subprocess, os
from pathlib import Path
from datetime import datetime

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return r.stdout.strip()

def analizar_logs(directorio):
    d = Path(directorio).expanduser().resolve()

    logs = list(d.glob("**/*.log"))
    if not logs:
        print("Non è stato trovato alcun file .log")
        return

    # Totale righe
    total_lineas = int(sh(f"cat {' '.join(str(l) for l in logs)} | wc -l"))

    # Ripartizione per livello
    niveles = {}
    for nivel in ("INFO", "WARNING", "ERROR"):
        count = sh(f"cat {' '.join(str(l) for l in logs)} | grep -c '{nivel}' || echo 0")
        niveles[nivel] = int(count)

    # Top 3 valori di loss
    top_loss_raw = sh(
        f"cat {' '.join(str(l) for l in logs)} | "
        f"grep -oE 'loss=[0-9.]+' | cut -d= -f2 | sort -rn | head -3"
    )

    # File con errori
    errores_por_archivo = {}
    for log in logs:
        n = sh(f"grep -c 'ERROR' {log} || echo 0")
        if int(n) > 0:
            errores_por_archivo[log.name] = int(n)

    # Costruire il report
    lineas = [
        f"REPORT DEI LOG — {d}",
        f"Generato: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"File analizzati: {len(logs)}",
        f"Totale righe: {total_lineas}",
        "",
        "RIPARTIZIONE PER LIVELLO:",
    ]
    for nivel, count in niveles.items():
        pct = (count / total_lineas * 100) if total_lineas else 0
        lineas.append(f"  {nivel:<10} {count:>4}  ({pct:.1f}%)")

    lineas += ["", "TOP 3 VALORI DI LOSS:"] + [
        f"  {v}" for v in top_loss_raw.splitlines()
    ]

    lineas += ["", "FILE CON ERRORI:"]
    if errores_por_archivo:
        for nombre, n in sorted(errores_por_archivo.items()):
            lineas.append(f"  {nombre}: {n} errore/i")
    else:
        lineas.append("  Nessuno")

    informe = "\n".join(lineas)
    print(informe)

    Path(d / "informe_logs.txt").write_text(informe + "\n", encoding="utf-8")
    print(f"\n✅ Report salvato in {d}/informe_logs.txt")


analizar_logs("~/busqueda_demo/logs")
```
</details>

---
## Riepilogo della lezione

```bash
# ─── grep ──────────────────────────────────────────────────
grep "pattern" file         # cercare nel file
grep -i "pattern" file      # ignorare maiuscole
grep -n "pattern" file      # con numero di riga
grep -r "pattern" dir/      # ricorsivo
grep -v "pattern" file      # invertire
grep -c "pattern" file      # contare
grep -E "regex" file        # espressioni regolari
grep -C 2 "pattern" file    # 2 righe di contesto

# ─── find ──────────────────────────────────────────────────
find . -name "*.py"         # per nome
find . -type f/d            # file / directory
find . -mtime -7            # modificati < 7 giorni fa
find . -size +1M            # maggiori di 1 MB
find . -empty               # vuoti
find . -name "*.tmp" -delete  # cercare ed eliminare
find . -name "*.py" -exec wc -l {} \;  # eseguire comando

# ─── Pipe e redirezioni ─────────────────────────────────────
cmd1 | cmd2                 # pipe
cmd > file                  # stdout → file (sovrascrivere)
cmd >> file                 # stdout → file (aggiungere)
cmd 2> errori                # stderr → file
cmd > tutto.txt 2>&1         # stdout + stderr → file
cmd 2>/dev/null             # sopprimere errori

# ─── Strumenti di testo ─────────────────────────────────────
wc -l file                  # contare righe
sort -n file                 # ordinare numericamente
sort | uniq                 # eliminare duplicati
uniq -c                     # contare ripetizioni
cut -d',' -f3               # estrarre colonna CSV
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 4** impariamo il **Bash scripting**: variabili, condizionali, cicli e funzioni. Con questo avremo i pezzi per il progetto finale di automazione.